# Week 4 — VQC Ablation Study

**Goal:** Find the best VQC depth and width for Week 5 final training.

| Config | Qubits | Layers | Description |
|--------|--------|--------|-------------|
| A1 | 3 | 1 | Too shallow? |
| A2 | 3 | 2 | Week 3 baseline (reproducibility check) |
| A3 | 3 | 3 | Barren plateau risk? |
| A4 | 5 | 2 | Richer Hilbert space |

Settings: `max_patches=1024`, `epochs=15`, `patience=5`, `seed=42`

In [1]:
import torch, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score
import json, time, sys, os, gc

# CRITICAL — prevents OOM fragmentation across epochs
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

sys.path.insert(0, '..')
from torch_geometric.loader import DataLoader as PyGLoader
from pathq.model_v2 import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT     = Path('..')
FEAT_DIR = Path('./data/features_uni')
CKPT_DIR = ROOT / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = ROOT / 'outputs';     OUT_DIR.mkdir(exist_ok=True)

MAX_PATCHES   = 512
EPOCHS        = 15
BATCH_SIZE    = 4
BATCH_SIZE_5Q = 2
LR            = 3e-5
PATIENCE      = 5
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device:   {DEVICE}')
print(f'GPU:      {torch.cuda.get_device_name(0)}')
print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Features: {len(list(FEAT_DIR.glob("*.pt")))} slides')
print(f'Patches:  {MAX_PATCHES} per slide (ablation setting)')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[model_v2] Using Transformer for global branch
Device:   cuda
GPU:      NVIDIA GeForce RTX 5060 Laptop GPU
VRAM:     8.1 GB
Features: 333 slides
Patches:  512 per slide (ablation setting)


In [2]:
train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = BATCH_SIZE,
    k            = 8,
    seed         = SEED,
    num_workers  = 0,
    max_patches  = MAX_PATCHES,
)

# Separate 5-qubit loaders with batch_size=2 to avoid OOM
train_loader_5q, val_loader_5q, test_loader_5q = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = BATCH_SIZE_5Q,
    k            = 8,
    seed         = SEED,
    num_workers  = 0,
    max_patches  = MAX_PATCHES,
)

print(f'Standard loaders (batch=4):  train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}')
print(f'5-qubit loaders  (batch=2):  train={len(train_loader_5q)} val={len(val_loader_5q)} test={len(test_loader_5q)}')

Split: train=233 (pos=78) val=50 (pos=17) test=50 (pos=16)
Split: train=233 (pos=78) val=50 (pos=17) test=50 (pos=16)
Standard loaders (batch=4):  train=59 val=13 test=13
5-qubit loaders  (batch=2):  train=117 val=25 test=25


In [3]:
def train_one(model, loader, opt, device):
    model.train()
    total, n = 0., 0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        torch.cuda.empty_cache()           # FIX 1 — clear before every batch
        logits, _ = model(batch)
        loss      = F.cross_entropy(logits, batch.y.view(-1))
        loss_val  = loss.item()            # FIX 2 — extract scalar BEFORE backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        del logits, loss                   # FIX 3 — free compute graph immediately
        torch.cuda.empty_cache()           # FIX 4 — clear after every batch
        total += loss_val
        n     += 1
    return total / max(n, 1)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    probs, labels, tl, n = [], [], 0., 0
    for batch in loader:
        batch     = batch.to(device)
        logits, _ = model(batch)
        tl       += F.cross_entropy(logits, batch.y.view(-1)).item()
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        labels.extend(batch.y.view(-1).cpu().tolist())
        n += 1
    p, l  = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp = ((preds==1)&(l==1)).sum(); fn = ((preds==0)&(l==1)).sum()
    tn = ((preds==0)&(l==0)).sum(); fp = ((preds==1)&(l==0)).sum()
    return {
        'auc':         auc,
        'f1':          f1,
        'loss':        tl / max(n, 1),
        'sensitivity': tp / max(tp+fn, 1),
        'specificity': tn / max(tn+fp, 1),
    }

def run_experiment(model, tr, va, te, device, label,
                   epochs=15, lr=3e-5, ckpt=None, patience=5):
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-3
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs, eta_min=1e-6
    )
    best_val_auc = 0.
    patience_ctr = 0

    print(f'\n{"─"*65}')
    print(f'  {"Ep":>3}  {"TrLoss":>8}  {"VaLoss":>8}  {"VaAUC":>8}  {"VaF1":>7}  {"Time":>7}')
    print(f'{"─"*65}')

    for ep in range(1, epochs + 1):
        t0      = time.time()
        tl      = train_one(model, tr, opt, device)
        vm      = eval_model(model, va, device)
        sched.step()
        elapsed = time.time() - t0
        flag    = ''

        if vm['auc'] > best_val_auc:
            best_val_auc = vm['auc']
            patience_ctr = 0
            flag         = '\u2713'
            if ckpt:
                torch.save({'model_state': model.state_dict(),
                            'epoch': ep, 'val_auc': best_val_auc}, ckpt)
        else:
            patience_ctr += 1

        overfit_warn = ' \u26a0 OVERFIT' if vm['loss'] > tl * 2.5 else ''

        print(f'  {ep:>3}  {tl:>8.4f}  {vm["loss"]:>8.4f}  '
              f'{vm["auc"]:>8.4f}  {vm["f1"]:>7.4f}  '
              f'{elapsed:>6.0f}s  {flag}{overfit_warn}')

        if patience_ctr >= patience:
            print(f'\n  Early stop at epoch {ep} (patience={patience})')
            break

        torch.cuda.empty_cache()

    if ckpt and Path(ckpt).exists():
        model.load_state_dict(
            torch.load(ckpt, weights_only=False)['model_state']
        )
    tm = eval_model(model, te, device)

    print(f'{"─"*65}')
    print(f'  Best val AUC: {best_val_auc:.4f}')
    print(f'  Test AUC:     {tm["auc"]:.4f}  '
          f'f1={tm["f1"]:.4f}  '
          f'sens={tm["sensitivity"]:.3f}  '
          f'spec={tm["specificity"]:.3f}')

    return {
        'val_auc':     best_val_auc,
        'auc':         tm['auc'],
        'f1':          tm['f1'],
        'sensitivity': tm['sensitivity'],
        'specificity': tm['specificity'],
        'gap':         tm['auc'] - best_val_auc,
    }

print('\u2705 Training functions ready')

✅ Training functions ready


In [4]:
ABLATIONS = [
    {
        'label':         'A1_vqc_1layer',
        'n_qubits':      3,
        'n_layers':      1,
        'desc':          '1 layer, 3 qubits — too shallow?',
        'use_5q_loader': False,
    },
    {
        'label':         'A2_vqc_2layer_base',
        'n_qubits':      3,
        'n_layers':      2,
        'desc':          '2 layers, 3 qubits — Week 3 baseline (reproducibility check)',
        'use_5q_loader': False,
    },
    {
        'label':         'A3_vqc_3layer',
        'n_qubits':      3,
        'n_layers':      3,
        'desc':          '3 layers, 3 qubits — barren plateau risk',
        'use_5q_loader': False,
    },
    {
        'label':         'A4_vqc_2layer_5qubit',
        'n_qubits':      5,
        'n_layers':      2,
        'desc':          '2 layers, 5 qubits — richer Hilbert space (batch=2)',
        'use_5q_loader': True,
    },
]

print(f'Week 4 ablation — {len(ABLATIONS)} configs')
print(f'max_patches={MAX_PATCHES}, epochs={EPOCHS}, patience={PATIENCE}, seed={SEED}')
print()
for a in ABLATIONS:
    bs = BATCH_SIZE_5Q if a['use_5q_loader'] else BATCH_SIZE
    print(f"  {a['label']}: {a['n_qubits']}q x {a['n_layers']}L  "
          f"batch={bs}  — {a['desc']}")

print()
print('ESTIMATED RUNTIME')
print(f'  A1 (1L-3Q):  ~25 min/epoch x 15 = ~6.3 hrs')
print(f'  A2 (2L-3Q):  ~38 min/epoch x 15 = ~9.5 hrs  (Week 3 baseline)')
print(f'  A3 (3L-3Q):  ~50 min/epoch x 15 = ~12.5 hrs')
print(f'  A4 (2L-5Q):  ~70 min/epoch x 15 = ~17.5 hrs  (batch=2)')
print(f'  Total:                             ~46 hrs (~2 days)')
print(f'  With early stopping (patience=5):  ~28-35 hrs (~1.5 days)')
print()
print('Run overnight. Checkpoints saved each epoch — safe to interrupt.')

Week 4 ablation — 4 configs
max_patches=512, epochs=15, patience=5, seed=42

  A1_vqc_1layer: 3q x 1L  batch=4  — 1 layer, 3 qubits — too shallow?
  A2_vqc_2layer_base: 3q x 2L  batch=4  — 2 layers, 3 qubits — Week 3 baseline (reproducibility check)
  A3_vqc_3layer: 3q x 3L  batch=4  — 3 layers, 3 qubits — barren plateau risk
  A4_vqc_2layer_5qubit: 5q x 2L  batch=2  — 2 layers, 5 qubits — richer Hilbert space (batch=2)

ESTIMATED RUNTIME
  A1 (1L-3Q):  ~25 min/epoch x 15 = ~6.3 hrs
  A2 (2L-3Q):  ~38 min/epoch x 15 = ~9.5 hrs  (Week 3 baseline)
  A3 (3L-3Q):  ~50 min/epoch x 15 = ~12.5 hrs
  A4 (2L-5Q):  ~70 min/epoch x 15 = ~17.5 hrs  (batch=2)
  Total:                             ~46 hrs (~2 days)
  With early stopping (patience=5):  ~28-35 hrs (~1.5 days)

Run overnight. Checkpoints saved each epoch — safe to interrupt.


In [5]:
# Run this BEFORE Cell 6 — confirms no crashes on any config
# Each test is <1 second. Saves you waking up to a crashed run.

from torch_geometric.data import Data, Batch

def quick_test(n_qubits, n_layers, label, batch_size=4):
    model = QuantaPathV2(
        use_vqc=True, n_qubits=n_qubits, vqc_layers=n_layers
    ).to(DEVICE)
    graphs = []
    for label_y in [1, 0]:
        g = Data(
            x          = torch.randn(20, 1040).to(DEVICE),
            edge_index = torch.randint(0, 20, (2, 60)).to(DEVICE),
            edge_attr  = torch.randn(60, 2).to(DEVICE),
            y          = torch.tensor([label_y]).to(DEVICE),
        )
        graphs.append(g)
    batch = Batch.from_data_list(graphs)
    with torch.no_grad():
        logits, _ = model(batch)
    assert logits.shape == (2, 2), f'Wrong shape: {logits.shape}'
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  \u2705 {label}: ({n_qubits}q, {n_layers}L) '
          f'-> {logits.shape}  params={n_params:,}')
    del model, batch
    torch.cuda.empty_cache()

print('Sanity checks — all 4 configs:')
quick_test(3, 1, 'A1_vqc_1layer')
quick_test(3, 2, 'A2_vqc_2layer_base')
quick_test(3, 3, 'A3_vqc_3layer')
quick_test(5, 2, 'A4_vqc_2layer_5qubit', batch_size=2)
print()
print('All configs OK — safe to run Cell 6 overnight.')

Sanity checks — all 4 configs:
[VQC] lightning.gpu (3q, 1L) — adjoint gradients
QuantaPathV2: use_vqc=True, trainable=970,379
  ✅ A1_vqc_1layer: (3q, 1L) -> torch.Size([2, 2])  params=970,379
[VQC] lightning.gpu (3q, 2L) — adjoint gradients
QuantaPathV2: use_vqc=True, trainable=970,385
  ✅ A2_vqc_2layer_base: (3q, 2L) -> torch.Size([2, 2])  params=970,385
[VQC] lightning.gpu (3q, 3L) — adjoint gradients
QuantaPathV2: use_vqc=True, trainable=970,391
  ✅ A3_vqc_3layer: (3q, 3L) -> torch.Size([2, 2])  params=970,391
[VQC] lightning.gpu (5q, 2L) — adjoint gradients
QuantaPathV2: use_vqc=True, trainable=973,467
  ✅ A4_vqc_2layer_5qubit: (5q, 2L) -> torch.Size([2, 2])  params=973,467

All configs OK — safe to run Cell 6 overnight.


In [6]:
ablation_results = {}

for cfg in ABLATIONS:
    print()
    print('='*65)
    print(f"  {cfg['label']}")
    print(f"  {cfg['n_qubits']} qubits x {cfg['n_layers']} layers")
    print(f"  {cfg['desc']}")
    print('='*65)

    if cfg['use_5q_loader']:
        tr, va, te = train_loader_5q, val_loader_5q, test_loader_5q
        print(f'  Using batch_size=2 loaders for 5-qubit VRAM safety')
    else:
        tr, va, te = train_loader, val_loader, test_loader

    model = QuantaPathV2(
        use_vqc    = True,
        n_qubits   = cfg['n_qubits'],
        vqc_layers = cfg['n_layers'],
    ).to(DEVICE)

    ckpt = str(CKPT_DIR / f"w4_{cfg['label']}_best.pth")

    result = run_experiment(
        model, tr, va, te, DEVICE,
        label    = cfg['label'],
        epochs   = EPOCHS,
        lr       = LR,
        ckpt     = ckpt,
        patience = PATIENCE,
    )

    ablation_results[cfg['label']] = {
        'n_qubits':    cfg['n_qubits'],
        'n_layers':    cfg['n_layers'],
        'val_auc':     result['val_auc'],
        'test_auc':    result['auc'],
        'f1':          result['f1'],
        'sensitivity': result['sensitivity'],
        'specificity': result['specificity'],
        'gap':         result['gap'],
    }

    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n  \u2705 {cfg['label']} done — "
          f"Test AUC: {result['auc']:.4f}  Gap: {result['gap']:+.4f}")

with open(OUT_DIR / 'week4_ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('\nAll 4 ablations complete.')
print('Saved: outputs/week4_ablation_results.json')


  A1_vqc_1layer
  3 qubits x 1 layers
  1 layer, 3 qubits — too shallow?
[VQC] lightning.gpu (3q, 1L) — adjoint gradients
QuantaPathV2: use_vqc=True, trainable=970,379

─────────────────────────────────────────────────────────────────
   Ep    TrLoss    VaLoss     VaAUC     VaF1     Time
─────────────────────────────────────────────────────────────────
    1    0.6832    0.6646    0.7094   0.5581    1209s  ✓
    2    0.6506    0.6314    0.6988   0.4375    1053s  
    3    0.6228    0.5870    0.7790   0.5625    1054s  ✓
    4    0.5827    0.5368    0.7932   0.5806    1054s  ✓
    5    0.5846    0.5209    0.8057   0.5625    1054s  ✓
    6    0.5520    0.4997    0.8111   0.6207    1055s  ✓
    7    0.5271    0.4645    0.8449   0.6897    1054s  ✓
    8    0.5058    0.4836    0.8200   0.6486    1059s  
    9    0.5226    0.4726    0.8164   0.5714    1052s  
   10    0.5192    0.4711    0.8164   0.6207    1053s  
   11    0.4904    0.4803    0.8128   0.7059    1053s  
   12    0.4960    0.4

In [7]:
print()
print('='*72)
print(f"{'Config':<24} {'Q':>3} {'L':>3} {'Val AUC':>9} "
      f"{'Test AUC':>9} {'F1':>7} {'Gap':>8} {'vs W3':>8}")
print('-'*72)

w3_test_auc = 0.7812   # Week 3 quantum baseline

for label, r in ablation_results.items():
    delta = r['test_auc'] - w3_test_auc
    print(f"{label:<24} {r['n_qubits']:>3} {r['n_layers']:>3} "
          f"{r['val_auc']:>9.4f} {r['test_auc']:>9.4f} "
          f"{r['f1']:>7.4f} {r['gap']:>+8.4f} {delta:>+8.4f}")

print('-'*72)
print(f"{'Classical GAT-Transformer':<24} {'-':>3} {'-':>3} "
      f"{'0.9537':>9} {'0.8382':>9} {'—':>7} {'-0.1155':>8} {'—':>8}")
print(f"{'Quantum W3 baseline':<24} {'3':>3} {'2':>3} "
      f"{'0.8217':>9} {'0.7812':>9} {'0.667':>7} {'-0.0405':>8} {'0.0000':>8}")

best = max(ablation_results.items(), key=lambda x: x[1]['test_auc'])
print(f'\nWinner: {best[0]}  '
      f'({best[1]["n_qubits"]}q, {best[1]["n_layers"]}L)  '
      f'Test AUC: {best[1]["test_auc"]:.4f}')
print(f'Use this config in Week 5 full training at max_patches=3000')


Config                     Q   L   Val AUC  Test AUC      F1      Gap    vs W3
------------------------------------------------------------------------
A1_vqc_1layer              3   1    0.8449    0.7132  0.4138  -0.1317  -0.0680
A2_vqc_2layer_base         3   2    0.8342    0.7610  0.4615  -0.0732  -0.0202
A3_vqc_3layer              3   3    0.7986    0.7463  0.5000  -0.0523  -0.0349
A4_vqc_2layer_5qubit       5   2    0.7968    0.7445  0.3478  -0.0523  -0.0367
------------------------------------------------------------------------
Classical GAT-Transformer   -   -    0.9537    0.8382       —  -0.1155        —
Quantum W3 baseline        3   2    0.8217    0.7812   0.667  -0.0405   0.0000

Winner: A2_vqc_2layer_base  (3q, 2L)  Test AUC: 0.7610
Use this config in Week 5 full training at max_patches=3000


In [8]:
best_label = max(ablation_results, key=lambda x: ablation_results[x]['test_auc'])
best       = ablation_results[best_label]

recommendation = {
    'best_config':       best_label,
    'optimal_n_qubits':  best['n_qubits'],
    'optimal_n_layers':  best['n_layers'],
    'ablation_test_auc': best['test_auc'],
    'ablation_patches':  MAX_PATCHES,
    'week3_test_auc':    0.7812,
    'improvement':       best['test_auc'] - 0.7812,
    'note': 'Retrain this config at max_patches=3000 in Week 5 for paper numbers',
}

with open(OUT_DIR / 'week4_best_config.json', 'w') as f:
    json.dump(recommendation, f, indent=2)

print('='*55)
print('WEEK 4 COMPLETE')
print('='*55)
print(f"Best config:     {best_label}")
print(f"n_qubits:        {best['n_qubits']}")
print(f"n_layers:        {best['n_layers']}")
print(f"Test AUC:        {best['test_auc']:.4f}  (at {MAX_PATCHES} patches)")
print(f"vs Week 3 (2L):  {best['test_auc'] - 0.7812:+.4f}")
print()
print('Next: Week 5 — retrain winner at max_patches=3000 on RunPod')
print('Saved: outputs/week4_best_config.json')

WEEK 4 COMPLETE
Best config:     A2_vqc_2layer_base
n_qubits:        3
n_layers:        2
Test AUC:        0.7610  (at 512 patches)
vs Week 3 (2L):  -0.0202

Next: Week 5 — retrain winner at max_patches=3000 on RunPod
Saved: outputs/week4_best_config.json
